# Suite TXT — Text shared by all implementations

Error and validation messages embed values the way Python's `repr` writes them, so every implementation must write
them identically. Python uses its built-ins (`repr`, `type(v).__name__`, `sorted`); TypeScript has its own
(`src/Framework/Repr.ts`). Both suites check the same expected text.

In [ ]:
import { pyFloat, repr, sortedStrings, Tuple, typeName } from "@mbse/schemas/Framework/Repr";
import { assert, same } from "./support.js";

## TXT-01 · Values are written as Python's `repr` writes them

In [ ]:
{
  const table: [unknown, string][] = [
    ["x", "'x'"],
    ["it's", "\"it's\""],
    ['a"b\'c', "'a\"b\\'c'"],
    ["\n\t\r\\", "'\\n\\t\\r\\\\'"],
    ["\x00\x1f\x7f", "'\\x00\\x1f\\x7f'"],
    ["\x85\xa0\xad", "'\\x85\\xa0\\xad'"],
    ["é日🚀", "'é日🚀'"],
    ["\ud800", "'\\ud800'"],
    [" ﻿​", "'\\u2028\\ufeff\\u200b'"],
    ["\u{e0001}", "'\\U000e0001'"],
    ["͸", "'\\ue000\\u0378'"],
    [null, "None"],
    [true, "True"],
    [false, "False"],
    [3n, "3"],
    [-(2n ** 100n), "-1267650600228229401496703205376"],
    [1.0, "1.0"],
    [-0.0, "-0.0"],
    [NaN, "nan"],
    [Infinity, "inf"],
    [-Infinity, "-inf"],
    [new Uint8Array(), "b''"],
    [new Uint8Array([0x00, 0xff, 0x27, 0x41]), 'b"\\x00\\xff\'A"'],
    [new Uint8Array([0x61, 0x22, 0x62]), "b'a\"b'"],
    [new Uint8Array([0x09, 0x0a, 0x0d, 0x5c]), "b'\\t\\n\\r\\\\'"],
    [new Uint8Array([0x27, 0x22]), "b'\\'\"'"],
    [["a", 1n], "['a', 1]"],
    [new Tuple(["a"]), "('a',)"],
    [new Tuple(["a", "b"]), "('a', 'b')"],
    [new Tuple([]), "()"],
    [new Map([["k", [1n]]]), "{'k': [1]}"],
    [new Set(), "set()"],
    [new Set(["a"]), "{'a'}"],
    [Array, "<class 'list'>"],
    [Map, "<class 'dict'>"],
    [BigInt, "<class 'int'>"],
    [String, "<class 'str'>"],
    [Uint8Array, "<class 'bytes'>"],
  ];
  for (const [value, expected] of table) assert(repr(value) === expected, `${repr(value)} !== ${expected}`);
}

## TXT-02 · Floats are written as Python's `float.__repr__` writes them

In [ ]:
{
  const floats: [number, string][] = [[0.0, "0.0"], [-0.0, "-0.0"], [1.0, "1.0"], [0.1, "0.1"], [1 / 3, "0.3333333333333333"],
    [1e16, "1e+16"], [1e15, "1000000000000000.0"], [123456789.125, "123456789.125"], [1e-4, "0.0001"], [1e-5, "1e-05"],
    [5e-324, "5e-324"], [1e308, "1e+308"], [2.5e-7, "2.5e-07"], [-1.5, "-1.5"], [1e21, "1e+21"],
    [1234567890123456.0, "1234567890123456.0"], [12345678901234567.0, "1.2345678901234568e+16"], [100.0, "100.0"],
    [0.001, "0.001"], [-2e-10, "-2e-10"]];
  for (const [value, expected] of floats) assert(pyFloat(value) === expected, `${pyFloat(value)} !== ${expected}`);
}

## TXT-03 · Type names use the framework's (Python's) vocabulary

In [ ]:
for (const [value, expected] of [[null, "NoneType"], [true, "bool"], [1n, "int"], [1.5, "float"], ["s", "str"],
  [new Uint8Array(), "bytes"], [[], "list"], [new Map(), "dict"], [new Set(), "set"], [{}, "object"], [() => null, "function"],
  [Object.create(null), "object"]] as const) { // the last row has no Python counterpart: an object without a prototype
  assert(typeName(value) === expected, `${typeName(value)} !== ${expected}`);
}

## TXT-04 · Strings sort by code point

In [ ]:
assert(same(sortedStrings(["b", "a", "é", "￿", "😀", "A", "Z", "_", "", "aa"]), ["", "A", "Z", "_", "a", "aa", "b", "é", "￿", "😀"]));